<a href="https://colab.research.google.com/github/GopalKrishna-India/Geospatial/blob/master/extract_haryana_tree_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install pdfplumber pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 40.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 33.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 61.8 MB/s eta 0:00:00
  Attempting uninstall: Pillow
    Found existing installation: pillow 11.3.0
    Uninstalling pillow-11.3.0:
      Successfully uninstalled pillow-11.3.0


In [ ]:
import os
import re
import glob
import pdfplumber
import pandas as pd


# ============================================================
# SETTINGS
# ============================================================

PDF_FOLDER = "/content/PDF_FOLDER"

OUTPUT_CSV = "/content/Haryana_Eucalyptus_Poplar_GT200.csv"

THRESHOLD = 200


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def clean_number(value):
    """
    Convert extracted table value to integer.
    Handles values such as:
        7800
        7,800
        '7800'
        None
    """
    if value is None:
        return None

    value = str(value).strip()

    # Remove commas, spaces and other non-numeric characters
    value = value.replace(",", "")
    value = re.sub(r"[^\d]", "", value)

    if value == "":
        return None

    return int(value)


def clean_village_name(name):
    """
    Clean village name extracted from the PDF header.
    """
    if not name:
        return None

    name = name.strip()

    # Remove unwanted spaces
    name = re.sub(r"\s+", " ", name)

    return name


def extract_village_names(page):
    """
    Extract village names from page text.

    Expected format:
        Forest Range: Chhachhrauli    Village: Arjun Majra

    Returns village names in their appearance order.
    """

    text = page.extract_text() or ""

    villages = []

    # Handle the usual format
    pattern = r"Village\s*:\s*(.*?)(?=\s{2,}Forest Range:|\s+Forest Range:|$)"

    matches = re.findall(pattern, text, flags=re.IGNORECASE)

    for match in matches:
        village = clean_village_name(match)

        if village:
            villages.append(village)

    # If the above pattern doesn't work, try line-by-line
    if not villages:

        for line in text.splitlines():

            match = re.search(
                r"Village\s*:\s*(.+)",
                line,
                flags=re.IGNORECASE
            )

            if match:
                village = match.group(1).strip()

                # Remove Forest Range portion if present
                village = re.split(
                    r"\s+Forest\s+Range\s*:",
                    village,
                    flags=re.IGNORECASE
                )[0].strip()

                if village:
                    villages.append(village)

    return villages


def identify_species_rows(table):
    """
    Find Eucalyptus and Poplar rows in an extracted PDF table.

    Returns:
        {
            "Eucalyptus": total,
            "Poplar": total
        }

    The last numeric value in each row is assumed to be the Total.
    """

    result = {
        "Eucalyptus": None,
        "Poplar": None
    }

    if not table:
        return result

    for row in table:

        if not row:
            continue

        # Convert cells to strings
        cells = [
            str(cell).strip() if cell is not None else ""
            for cell in row
        ]

        row_text = " ".join(cells)

        # ----------------------------------------------------
        # Eucalyptus
        # ----------------------------------------------------

        if re.search(r"\bEucalyptus\b", row_text, re.IGNORECASE):

            numbers = []

            for cell in cells[1:]:
                number = clean_number(cell)

                if number is not None:
                    numbers.append(number)

            if numbers:
                result["Eucalyptus"] = numbers[-1]

        # ----------------------------------------------------
        # Poplar
        # ----------------------------------------------------

        elif re.search(r"\bPoplar\b", row_text, re.IGNORECASE):

            numbers = []

            for cell in cells[1:]:
                number = clean_number(cell)

                if number is not None:
                    numbers.append(number)

            if numbers:
                result["Poplar"] = numbers[-1]

    return result


def table_is_tree_table(table):
    """
    Check whether an extracted table is the
    'Number of Trees Outside Forests' table.
    """

    if not table:
        return False

    text = " ".join(
        str(cell)
        for row in table
        if row
        for cell in row
        if cell is not None
    )

    return (
        re.search(r"Eucalyptus", text, re.IGNORECASE) is not None
        or
        re.search(r"Poplar", text, re.IGNORECASE) is not None
    )


# ============================================================
# PROCESS ONE PDF
# ============================================================

def process_pdf(pdf_path):

    district = os.path.splitext(
        os.path.basename(pdf_path)
    )[0]

    records = []

    print(f"\nProcessing: {os.path.basename(pdf_path)}")

    with pdfplumber.open(pdf_path) as pdf:

        for page_number, page in enumerate(pdf.pages, start=1):

            # ------------------------------------------------
            # Extract village names on this page
            # ------------------------------------------------

            villages = extract_village_names(page)

            # ------------------------------------------------
            # Extract tables
            # ------------------------------------------------

            try:
                tables = page.extract_tables()
            except Exception as e:
                print(
                    f"  Page {page_number}: "
                    f"table extraction error: {e}"
                )
                continue

            tree_tables = []

            for table in tables:

                if table_is_tree_table(table):
                    tree_tables.append(table)

            if not tree_tables:
                continue

            # ------------------------------------------------
            # Match villages with tree tables
            # ------------------------------------------------

            for i, table in enumerate(tree_tables):

                species = identify_species_rows(table)

                eucalyptus = species["Eucalyptus"]
                poplar = species["Poplar"]

                # Skip if neither species could be extracted
                if eucalyptus is None and poplar is None:
                    continue

                # Try to associate table with village
                if i < len(villages):
                    village = villages[i]
                else:
                    village = f"UNKNOWN_VILLAGE_PAGE_{page_number}_{i+1}"

                # Missing species count = 0
                if eucalyptus is None:
                    eucalyptus = 0

                if poplar is None:
                    poplar = 0

                # ------------------------------------------------
                # QUALIFICATION CRITERION
                #
                # Include if EITHER species > 200
                # ------------------------------------------------

                if eucalyptus > THRESHOLD or poplar > THRESHOLD:

                    qualifying_species = []

                    if eucalyptus > THRESHOLD:
                        qualifying_species.append("Eucalyptus")

                    if poplar > THRESHOLD:
                        qualifying_species.append("Poplar")

                    records.append({
                        "District": district,
                        "Village": village,
                        "Eucalyptus_Trees": eucalyptus,
                        "Poplar_Trees": poplar,
                        "Qualifying_Species": "; ".join(
                            qualifying_species
                        ),
                        "Source_PDF": os.path.basename(pdf_path),
                        "Page": page_number
                    })

                    print(
                        f"  Page {page_number}: "
                        f"{village} | "
                        f"Eucalyptus={eucalyptus:,} | "
                        f"Poplar={poplar:,}"
                    )

    return records


# ============================================================
# MAIN
# ============================================================

def main():

    pdf_files = sorted(
        glob.glob(
            os.path.join(PDF_FOLDER, "*.pdf")
        )
    )

    if not pdf_files:
        print("No PDF files found.")
        return

    print("=" * 70)
    print(f"Found {len(pdf_files)} PDF files")
    print("=" * 70)

    all_records = []

    for pdf_path in pdf_files:

        try:
            records = process_pdf(pdf_path)
            all_records.extend(records)

        except Exception as e:

            print(
                f"\nERROR processing "
                f"{os.path.basename(pdf_path)}:"
            )
            print(e)

    # ========================================================
    # CREATE DATAFRAME
    # ========================================================

    df = pd.DataFrame(all_records)

    if df.empty:

        print("\nNo qualifying villages were found.")
        return

    # --------------------------------------------------------
    # Sort
    # --------------------------------------------------------

    df = df.sort_values(
        by=[
            "District",
            "Village"
        ]
    )

    # --------------------------------------------------------
    # Remove exact duplicates if any
    # --------------------------------------------------------

    df = df.drop_duplicates(
        subset=[
            "District",
            "Village",
            "Eucalyptus_Trees",
            "Poplar_Trees"
        ]
    )

    # ========================================================
    # SAVE CSV
    # ========================================================

    df.to_csv(
        OUTPUT_CSV,
        index=False,
        encoding="utf-8-sig"
    )

    # ========================================================
    # SUMMARY
    # ========================================================

    print("\n" + "=" * 70)
    print("EXTRACTION COMPLETE")
    print("=" * 70)

    print(f"PDFs processed       : {len(pdf_files)}")
    print(f"Qualifying villages  : {len(df)}")
    print(f"Output CSV           : {OUTPUT_CSV}")

    print("\nDistrict-wise village count:")
    print(
        df.groupby("District")
        .size()
        .sort_index()
    )

    print("\nSample output:")
    print(
        df.head(20).to_string(index=False)
    )


if __name__ == "__main__":
    main()

Found 22 PDF files

Processing: Ambala.pdf
  Page 8: UNKNOWN_VILLAGE_PAGE_8_2 | Eucalyptus=8,013 | Poplar=2
  Page 9: UNKNOWN_VILLAGE_PAGE_9_1 | Eucalyptus=0 | Poplar=770
  Page 9: UNKNOWN_VILLAGE_PAGE_9_2 | Eucalyptus=17,610 | Poplar=19
  Page 11: Babyal | Eucalyptus=91 | Poplar=453
  Page 12: Balana | Eucalyptus=809 | Poplar=951
  Page 13: Baldev Camp | Eucalyptus=12 | Poplar=2,592
  Page 14: Bara | Eucalyptus=2 | Poplar=348
  Page 15: Baranala | Eucalyptus=3 | Poplar=1,055
  Page 15: Baroul Barouli | Eucalyptus=6 | Poplar=251
  Page 17: Batrohan | Eucalyptus=0 | Poplar=322
  Page 19: Bhapur Nakatpur | Eucalyptus=1,381 | Poplar=0
  Page 19: Bhari | Eucalyptus=2,440 | Poplar=760
  Page 20: Bharinga | Eucalyptus=851 | Poplar=0
  Page 20: Bhunni | Eucalyptus=744 | Poplar=0
  Page 21: Bichpadi | Eucalyptus=4,005 | Poplar=9,862
  Page 22: Bishangarh | Eucalyptus=622 | Poplar=0
  Page 22: Boh | Eucalyptus=572 | Poplar=240
  Page 23: Bohawa | Eucalyptus=5,184 | Poplar=362
  Page 23: Brahman